# Week 7 - Task 1: Advanced Feature Engineering

## Objective

This notebook demonstrates how polynomial features, interaction terms, and mathematical transformations can expose nonlinear patterns to a machine-learning model.

We use the **UCI Wine Quality - Red** tabular dataset and a regression formulation where `quality` is the target.

### Workflow
1. Load and inspect the raw dataset.
2. Clean the data and establish a baseline Linear Regression model.
3. Add polynomial features and interaction terms.
4. Apply custom mathematical transformations to skewed numerical features.
5. Retrain and compare performance.
6. Inspect feature distributions and coefficients.
7. Save reproducible metrics and a feature-engineering impact table.

### Main evaluation metrics
- MAE: lower is better
- RMSE: lower is better
- R²: higher is better

## Dataset

**Dataset:** Wine Quality - Red  
**Source:** UCI Machine Learning Repository  
**Task:** Predict wine quality from physicochemical measurements.

The dataset contains 1,599 red-wine samples and 11 physicochemical input variables. The target is the integer `quality` score.

The notebook downloads the original semicolon-separated CSV from UCI when run. If internet access is unavailable, place `winequality-red.csv` in the `data/` folder.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_DIR = "../data"
DATA_PATH = os.path.join(DATA_DIR, "winequality-red.csv")

os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
# Download the UCI dataset if it is not already present.
if not os.path.exists(DATA_PATH):
    import urllib.request
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
    urllib.request.urlretrieve(url, DATA_PATH)

df = pd.read_csv(DATA_PATH, sep=";")

print("Shape:", df.shape)
display(df.head())

In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing"))

print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nSummary statistics:")
display(df.describe().T)

## 1. Initial Cleaning

The UCI dataset is already mostly clean, but we still:
- remove duplicate rows,
- check for missing values,
- separate features and target,
- use a fixed train/test split for reproducibility.

We do not transform the test set independently; all learned preprocessing is fitted only on training data through Scikit-Learn pipelines.

In [ ]:
df_clean = df.drop_duplicates().copy()
df_clean = df_clean.dropna()

X = df_clean.drop(columns=["quality"])
y = df_clean["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Cleaned shape:", df_clean.shape)
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

## 2. Baseline Model

A Linear Regression model is used as the baseline. Standardization is included in the pipeline so the coefficients are on comparable scales.

The baseline represents what a simple linear relationship can achieve before explicit nonlinear feature engineering.

In [ ]:
baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

baseline_metrics = regression_metrics(y_test, baseline_pred)
baseline_metrics

In [ ]:
baseline_results = pd.DataFrame([baseline_metrics], index=["Baseline Linear Regression"])
display(baseline_results)

## 3. Polynomial Features and Interaction Terms

Polynomial features let the model represent terms such as:

- `x²`
- `x³`
- `x₁ × x₂`

For this experiment, degree 2 is used and `include_bias=False` avoids adding a redundant constant column.

`interaction_only=False` means the transformer includes both squared terms and pairwise interactions.

In [ ]:
poly_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

poly_model.fit(X_train, y_train)
poly_pred = poly_model.predict(X_test)

poly_metrics = regression_metrics(y_test, poly_pred)
poly_metrics

In [ ]:
poly_feature_count = poly_model.named_steps["poly"].n_output_features_
print("Original features:", X_train.shape[1])
print("Polynomial/interaction features:", poly_feature_count)

poly_results = pd.DataFrame([poly_metrics], index=["Polynomial + Interactions"])
display(poly_results)

## 4. Custom Mathematical Transformations

Many physicochemical variables are positive and can be right-skewed. A log transform can compress large values and sometimes make a relationship easier for a linear model to learn.

We apply `log1p(x) = log(1+x)` to selected positive features:

- `residual sugar`
- `chlorides`
- `free sulfur dioxide`
- `total sulfur dioxide`

`log1p` is safe for zero-valued observations.

The transformations are generated from the training/test data separately using a reusable function. Because these transforms do not learn parameters from the data, there is no train/test leakage.

In [ ]:
LOG_FEATURES = [
    "residual sugar",
    "chlorides",
    "free sulfur dioxide",
    "total sulfur dioxide",
]

def add_log_features(frame, columns):
    result = frame.copy()
    for col in columns:
        # These variables are non-negative in the dataset.
        result[f"log1p_{col}"] = np.log1p(result[col].clip(lower=0))
    return result

X_train_log = add_log_features(X_train, LOG_FEATURES)
X_test_log = add_log_features(X_test, LOG_FEATURES)

log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

log_model.fit(X_train_log, y_train)
log_pred = log_model.predict(X_test_log)

log_metrics = regression_metrics(y_test, log_pred)
log_metrics

In [ ]:
log_results = pd.DataFrame([log_metrics], index=["Log-transformed Features"])
display(log_results)

## 5. Combined Feature Engineering

The strongest experiment combines:
- original features,
- log-transformed versions of selected skewed variables,
- degree-2 polynomial terms and interactions.

Ridge regression is used after polynomial expansion to reduce instability caused by correlated engineered features.

In [ ]:
combined_train = add_log_features(X_train, LOG_FEATURES)
combined_test = add_log_features(X_test, LOG_FEATURES)

combined_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10.0))
])

combined_model.fit(combined_train, y_train)
combined_pred = combined_model.predict(combined_test)

combined_metrics = regression_metrics(y_test, combined_pred)
combined_metrics

In [ ]:
all_results = pd.DataFrame(
    [baseline_metrics, poly_metrics, log_metrics, combined_metrics],
    index=[
        "Baseline Linear Regression",
        "Polynomial + Interactions",
        "Log-transformed Features",
        "Combined Engineering"
    ]
)

display(all_results.round(4))

## 6. Performance Impact

The following table compares each technique with the baseline.

For MAE and RMSE, a positive percentage means the error decreased, which is an improvement.

For R², a positive percentage means explained variance increased.

In [ ]:
baseline_mae = baseline_metrics["MAE"]
baseline_rmse = baseline_metrics["RMSE"]
baseline_r2 = baseline_metrics["R2"]

impact = all_results.copy()
impact["MAE_change_%"] = (baseline_mae - impact["MAE"]) / baseline_mae * 100
impact["RMSE_change_%"] = (baseline_rmse - impact["RMSE"]) / baseline_rmse * 100
impact["R2_change_%"] = (impact["R2"] - baseline_r2) / max(abs(baseline_r2), 1e-12) * 100

display(impact.round(4))

os.makedirs("../docs", exist_ok=True)
impact.to_csv("../docs/performance_comparison.csv")

## 7. Feature Distribution Inspection

Visual inspection helps explain why transformations can help. Highly skewed variables can have a long tail that disproportionately influences a linear model.

The plots below compare selected raw variables with their `log1p` versions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, col in zip(axes, LOG_FEATURES):
    ax.hist(X_train[col], bins=30)
    ax.set_title(col)
    ax.set_xlabel("Raw value")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("../docs/raw_feature_distributions.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, col in zip(axes, LOG_FEATURES):
    ax.hist(np.log1p(X_train[col].clip(lower=0)), bins=30)
    ax.set_title(f"log1p({col})")
    ax.set_xlabel("Transformed value")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("../docs/log_feature_distributions.png", dpi=150)
plt.show()

## 8. Actual vs Predicted Values

The closer the points are to the diagonal line, the closer the predictions are to the observed quality scores.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, baseline_pred, alpha=0.5, label="Baseline")
ax.scatter(y_test, combined_pred, alpha=0.5, label="Combined")
lims = [min(y_test.min(), baseline_pred.min(), combined_pred.min()),
        max(y_test.max(), baseline_pred.max(), combined_pred.max())]
ax.plot(lims, lims, linestyle="--")
ax.set_xlabel("Actual quality")
ax.set_ylabel("Predicted quality")
ax.set_title("Actual vs Predicted Wine Quality")
ax.legend()
plt.tight_layout()
plt.savefig("../docs/actual_vs_predicted.png", dpi=150)
plt.show()

## 9. Interpretation

Use the computed `impact` table rather than assuming that feature engineering always improves performance.

### Polynomial and interaction features
These allow a linear model to approximate nonlinear relationships. The trade-off is a larger feature space, stronger correlations between features, and a greater risk of overfitting. Ridge regularization helps control this.

### Log transformations
A logarithm compresses large values and reduces the influence of long right tails. This can improve a model when the target relationship is closer to multiplicative or when extreme observations dominate the raw scale.

### Combined model
The combined model has the greatest representational flexibility. Its test-set metrics determine whether that extra flexibility actually improves generalization.

### Reproducibility
All train/test splits use `random_state=42`. The UCI source is recorded in `data/raw_dataset_info.txt`, and the generated comparison table is saved to `docs/performance_comparison.csv`.

In [ ]:
# Save a compact machine-readable summary.
summary = {
    "dataset": "UCI Wine Quality - Red",
    "rows_after_cleaning": int(len(df_clean)),
    "original_features": int(X.shape[1]),
    "polynomial_features_degree_2": int(poly_feature_count),
    "baseline": baseline_metrics,
    "polynomial_interactions": poly_metrics,
    "log_transforms": log_metrics,
    "combined": combined_metrics,
}

with open("../docs/metrics_summary.json", "w", encoding="utf-8") as f:
    import json
    json.dump(summary, f, indent=2)

print("Saved:")
print("- ../docs/performance_comparison.csv")
print("- ../docs/metrics_summary.json")
print("- ../docs/raw_feature_distributions.png")
print("- ../docs/log_feature_distributions.png")
print("- ../docs/actual_vs_predicted.png")